In [1]:
# Supervisor Pattern cơ bản
import os
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

True

In [2]:
# State
class TrangThaiDuAn(TypedDict):
    messages: Annotated[list, add_messages]
    agent_hien_tai: str
    ket_qua_nghien_cuu: str
    ket_qua_phan_tich: str
    ket_qua_de_xuat: str
    so_vong: int

In [3]:
# Structure output cho Supervisor
class QuyetDinhSupervisor(BaseModel):
    agent_tiep_theo: Literal[
        "agent_nghien_cuu",
        "agent_phan_tich",
        "agent_de_xuat",
        "HOAN_THANH"
    ]
    ly_do: str
    huong_dan_cu_the: str

In [4]:
# Khoi tao LLM
llm_supervisor = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
).with_structured_output(QuyetDinhSupervisor)

llm_worker = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

In [5]:
# Node Supervisor
def supervisor(state: TrangThaiDuAn) -> dict:
    """
    Supervisor đọc toàn bộ context và quyết định bước tiếp theo.
    Đây là 'bộ não' điều phối toàn bộ workflow.
    """
    so_vong = state.get("so_vong", 0)
    if so_vong >= 6:
        return {
            "agent_hien_tai": "HOAN_THANH",
            "so_vong": so_vong
        }
    
    tom_tat_hien_tai = tom_tat_hien_tai = f"""
Kết quả nghiên cứu: {state.get('ket_qua_nghien_cuu', 'Chưa có')}
Kết quả phân tích: {state.get('ket_qua_phan_tich', 'Chưa có')}
Đề xuất chiến lược: {state.get('ket_qua_de_xuat', 'Chưa có')}
Số vòng đã chạy: {so_vong}
"""

    system_prompt = """Bạn là Supervisor điều phối một nhóm agent tư vấn kinh doanh cho doanh nghiệp Việt Nam.

Nhóm của bạn gồm:
- agent_nghien_cuu: Nghiên cứu thị trường, đối thủ cạnh tranh, xu hướng ngành
- agent_phan_tich: Phân tích số liệu, đánh giá rủi ro, SWOT analysis  
- agent_de_xuat: Đưa ra chiến lược và kế hoạch hành động cụ thể

Quy tắc điều phối:
1. Bắt đầu luôn với agent_nghien_cuu để có thông tin nền
2. Chỉ chuyển sang agent_phan_tich sau khi có kết quả nghiên cứu
3. Chỉ chuyển sang agent_de_xuat sau khi có phân tích
4. Chọn HOAN_THANH khi đã có đủ ba phần và chất lượng tốt

Hãy phân tích trạng thái hiện tại và quyết định bước tiếp theo."""

    yeu_cau_nguoi_dung = state["messages"][0].content

    ket_qua = llm_supervisor.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
Yêu cầu gốc từ người dùng: {yeu_cau_nguoi_dung}

Trạng thái hiện tại:
{tom_tat_hien_tai}

Quyết định bước tiếp theo:""")
    ])

    return {
        "agent_hien_tai": ket_qua.agent_tiep_theo,
        "so_vong": so_vong + 1,
        "messages": [AIMessage(content=f"[Supervisor] → {ket_qua.agent_tiep_theo}: {ket_qua.ly_do}")]
    }

In [6]:
# Ba Worker Nodes
def agent_nghien_cuu(state: TrangThaiDuAn) -> dict:
    """Agent chuyên nghiên cứu thị trường."""
    yeu_cau = state["messages"][0].content
    agent_hien_tai = state.get("agent_hien_tai", "")

    huong_dan = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage) and "[Supervisor]" in msg.content:
            huong_dan = msg.content
            break

    system = """Bạn là chuyên gia nghiên cứu thị trường Việt Nam với 10 năm kinh nghiệm.
Nhiệm vụ: Cung cấp thông tin thị trường chính xác, cập nhật, và có giá trị thực tiễn.
Tập trung vào: Quy mô thị trường, xu hướng, đối thủ chính, cơ hội và thách thức.
Trả lời bằng tiếng Việt, có cấu trúc rõ ràng, dùng số liệu cụ thể khi có thể."""

    ket_qua = llm_worker.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Nghiên cứu thị trường cho yêu cầu sau: {yeu_cau}")
    ])

    return {
        "ket_qua_nghien_cuu": ket_qua.content,
        "messages": [AIMessage(content=f"[Nghiên cứu]\n{ket_qua.content}")]
    }

def agent_phan_tich(state: TrangThaiDuAn) -> dict:
    """Agent chuyên phân tích và đánh giá."""
    yeu_cau = state["messages"][0].content
    nghien_cuu = state.get("ket_qua_nghien_cuu", "Chưa có dữ liêu nghiên cứu")

    system = """Bạn là chuyên gia phân tích kinh doanh.
Nhiệm vụ: Phân tích sâu dữ liệu đã có, đánh giá rủi ro và cơ hội, thực hiện SWOT.
Dựa trên dữ liệu nghiên cứu được cung cấp — không bịa đặt số liệu.
Trả lời có cấu trúc: Điểm mạnh / Điểm yếu / Cơ hội / Thách thức."""

    ket_qua = llm_worker.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"""
Yêu cầu phân tích: {yeu_cau}

Dữ liệu nghiên cứu có sẵn:
{nghien_cuu}

Hãy phân tích chi tiết:""")
    ])

    return {
        "ket_qua_phan_tich": ket_qua.content,
        "messages": [AIMessage(content=f"[Phân Tích]\n{ket_qua.content}")]
    }

def agent_de_xuat(state: TrangThaiDuAn) -> dict:
    """Agent chuyên đưa ra đề xuất chiến lược."""
    yeu_cau = state["messages"][0].content
    nghien_cuu = state.get("ket_qua_nghien_cuu", "")
    phan_tich = state.get("ket_qua_phan_tich", "")

    system = """Bạn là tư vấn chiến lược kinh doanh cao cấp.
Nhiệm vụ: Dựa trên nghiên cứu và phân tích, đưa ra kế hoạch hành động cụ thể.
Đề xuất phải: Thực tế, có thể thực hiện được trong bối cảnh Việt Nam.
Cấu trúc: Chiến lược tổng thể → Các bước cụ thể → Timeline → KPI đo lường."""

    ket_qua = llm_worker.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"""
Yêu cầu: {yeu_cau}

Nghiên cứu thị trường:
{nghien_cuu}

Phân tích:
{phan_tich}

Đề xuất chiến lược và kế hoạch hành động:""")
    ])

    return {
        "ket_qua_de_xuat": ket_qua.content,
        "messages": [AIMessage(content=f"[Đề Xuất Chiến Lược]\n{ket_qua.content}")]
    }

In [7]:
# routing
def dinh_tuyen_supervisor(state: TrangThaiDuAn) -> str:
    """Đọc quyết định của Supervisor và trả về tên Node tiếp theo."""
    agent = state.get("agent_hien_tai", "HOAN_THANH")
    if agent == "HOAN_THANH":
        return END
    return agent

# Build graph
graph_builder = StateGraph(TrangThaiDuAn)

graph_builder.add_node("supervisor", supervisor)
graph_builder.add_node("agent_nghien_cuu", agent_nghien_cuu)
graph_builder.add_node("agent_phan_tich", agent_phan_tich)
graph_builder.add_node("agent_de_xuat", agent_de_xuat)

graph_builder.add_edge(START, "supervisor")
graph_builder.add_conditional_edges(
    "supervisor",
    dinh_tuyen_supervisor,
    {
        "agent_nghien_cuu": "agent_nghien_cuu",
        "agent_phan_tich": "agent_phan_tich",
        "agent_de_xuat": "agent_de_xuat",
        END: END
    }
)

graph_builder.add_edge("agent_nghien_cuu", "supervisor")
graph_builder.add_edge("agent_phan_tich", "supervisor")
graph_builder.add_edge("agent_de_xuat", "supervisor")

memory = MemorySaver()
graph_supervisor = graph_builder.compile(checkpointer=memory)

In [8]:
# Chay thu
def chay_supervisor(yeu_cau: str, thread_id: str = "test_01"):
    config = {"configurable": {"thread_id": thread_id}}
    state_dau_vao = {
        "messages": [HumanMessage(content=yeu_cau)],
        "agent_hien_tai": "",
        "ket_qua_nghien_cuu": "",
        "ket_qua_phan_tich": "",
        "ket_qua_de_xuat": "",
        "so_vong": 0
    }

    print(f"\nYêu cầu: {yeu_cau}")

    ket_qua = graph_supervisor.invoke(state_dau_vao, config=config)

    print(f"Nghiên cứu:\n{ket_qua['ket_qua_nghien_cuu'][:300]}...")
    print(f"\nPhân tích:\n{ket_qua['ket_qua_phan_tich'][:300]}...")
    print(f"\nĐề xuất:\n{ket_qua['ket_qua_de_xuat'][:300]}...")
    print(f"\nSố vòng đã chạy: {ket_qua['so_vong']}")


chay_supervisor(
    "Tôi muốn mở quán cà phê tại Đà Nẵng, nhắm vào khách du lịch nước ngoài. "
    "Vốn ban đầu khoảng 500 triệu. Tôi nên làm gì?"
)


Yêu cầu: Tôi muốn mở quán cà phê tại Đà Nẵng, nhắm vào khách du lịch nước ngoài. Vốn ban đầu khoảng 500 triệu. Tôi nên làm gì?
Nghiên cứu:
Mở quán cà phê tại Đà Nẵng, nhắm vào khách du lịch nước ngoài là một ý tưởng hấp dẫn, nhất là trong bối cảnh thành phố này đang phát triển mạnh mẽ về du lịch. Dưới đây là một số thông tin và gợi ý cụ thể để bạn có thể thực hiện kế hoạch của mình.

### 1. Quy mô thị trường
- **Khách du lịch**: Đà Nẵn...

Phân tích:
### Phân Tích SWOT Cho Kế Hoạch Mở Quán Cà Phê Tại Đà Nẵng Nhắm Vào Khách Du Lịch Nước Ngoài

#### Điểm mạnh (Strengths)
1. **Thị trường du lịch lớn**: Đà Nẵng là điểm đến du lịch nổi tiếng với lượng khách quốc tế đáng kể, tạo cơ hội lớn cho quán cà phê phục vụ đối tượng này.
2. **Xu hướng tiêu dùng...

Đề xuất:
### Chiến lược tổng thể
Mở quán cà phê tại Đà Nẵng nhắm đến khách du lịch nước ngoài với mô hình kinh doanh độc đáo, tập trung vào chất lượng sản phẩm, trải nghiệm khách hàng, và sử dụng các kênh truyền thông xã hội để quảng bá th